# 8. Frozen prospective, shifted-world, and boundary-condition validation

## Goal

After the intervention library, candidate budgets, model choice, and evaluation protocol are frozen, does the workflow retain a decision advantage in prospective and harder-world tests?

This notebook is a readable research record. It follows the actual hand-offs in order and loads the saved evidence by default; it does **not** hide the experiment behind a one-cell runner.


## Pipeline at a glance

```text
goal → declared generator → observable/lockbox split → model setup & training
     → candidate or condition screen → matched comparison → interpretation
```

Each section below corresponds to one of these hand-offs.


In [ ]:
# Run this notebook from the repository root.
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

REGENERATE = False  # Cached artifacts are the default; no expensive solve runs implicitly.

def artifact(relative_path: str) -> Path:
    """Fail with a useful message rather than silently replacing evidence."""
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing cached artifact: {path}")
    return path

def show(frame, n=8):
    # `print` keeps this notebook usable in a plain Python kernel as well as Jupyter.
    print(frame.head(n).to_string(index=False))
    print(f"{len(frame):,} rows × {len(frame.columns):,} columns")


## 1. Experimental contract

The experiment has a declared observation boundary. “Observable” means the learner may use it; “lockbox” means it may be generated and audited but must not be used as a deployable feature.


In [ ]:
from yeast_validation import run_final_validation_prospective as final
from yeast_validation import run_prospective_dbtl_benchmark as prospective

print("final evaluator:", final.__name__)
print("prospective campaign definition:", prospective.__name__)
print("freeze boundary: candidate library, budgets, checkpoints, and outcome metric")


## 2. Data generator

Prospective campaigns keep candidate selection separate from the exact verifier. Data-sufficiency and shifted-world suites alter the declared training amount or world, not the outcome metric after the fact.

The next cell exposes the generator’s first concrete hand-off. It is deliberately small/inspection-only where generating the full campaign is expensive.


In [ ]:
# The final phase begins with a frozen campaign definition, not a newly optimized pool.
summary = pd.read_csv(artifact("data/final_validation_prospective/final_validation_predictive_overfitting_summary.csv"))
show(summary)


## 3. Model setup and training contract

The selected hybrid and conventional routes operate under their predeclared data and candidate interfaces. Exact replay remains the arbiter for candidate-level claims.

Training is not automatically started in this notebook. The cached training/evaluation artifacts below are the evidence record; regeneration must be an intentional, parameterized action.


In [ ]:
# Make the experiment hand-off inspectable before looking at aggregate metrics.
for name, relative_path in [('summary', 'data/final_validation_prospective/final_validation_predictive_overfitting_summary.csv')]:
    path = artifact(relative_path)
    print(f"{name}: {path.relative_to(ROOT)}")


## 4. Screening / selection stage

Generate rankings from frozen checkpoints, select the allowed shortlist, then verify that shortlist using the declared real-GSM/hidden-world evaluator.

The screen is intentionally shown separately from final verification, so a virtual score cannot be mistaken for an exact outcome.


In [ ]:
# Load the primary evidence table and inspect its schema before aggregation.
summary = pd.read_csv(artifact('data/final_validation_prospective/final_validation_predictive_overfitting_summary.csv'))
show(summary)


## 5. Matched comparison

Compare prospective campaigns, training-set sizes, and world shifts using exact outcomes and declared accounting—not only in-sample predictive fit.


In [ ]:
# Aggregate only over fields that exist in this version of the cached record.
comparison = summary.groupby('training_cultures_n', dropna=False).mean(numeric_only=True)
show(comparison.reset_index() if hasattr(comparison, "reset_index") else comparison)


## 6. Analysis view

The plot is intentionally generic: it exposes every numeric evidence column so the reader can select the metric relevant to the claim, rather than hard-coding an attractive subset.


In [ ]:
numeric = summary.select_dtypes("number")
if numeric.shape[1]:
    ax = numeric.plot(kind="box", rot=45, figsize=(11, 4), title="Cached evidence: numeric metric distribution")
    ax.set_ylabel("recorded metric value")
    plt.tight_layout()
else:
    print("This artifact has no numeric columns to plot.")


## 7. Interpretation, scope, and next hand-off

This is the strongest but still bounded result: it supports conditional computational performance inside these generators, budgets, and exact checks—not wet-lab transfer.

### Reproduction boundary

The cells above reveal the inputs and artifacts without launching an expensive campaign. To regenerate, use the explicit command below only after reviewing its declared inputs and output destination.


In [ ]:
if REGENERATE:
    # This guard prevents accidental solver/campaign execution.
    raise RuntimeError('Final prospective validation has frozen campaign inputs and distributed exact verification. Use the recorded campaign manifests rather than rerunning a hidden wrapper.')
